# Cryptocurrency Price Direction Prediction Using Machine Learning

## Project Overview

**Research Question:** Can machine learning models predict the next-day price direction of Bitcoin and Ethereum using technical indicators?

**Target Audience:** Cryptocurrency investors, quantitative analysts, and finance students interested in algorithmic trading signals.

**Dataset:** Daily OHLCV data for BTC-USD and ETH-USD (2022–2025), sourced from Yahoo Finance via the `yfinance` library. Accessed: April 2026.

**Methods:** Feature engineering (technical indicators), Random Forest Classifier, Logistic Regression, model evaluation and comparison.

**Author:** Meiya Han | ACC102 Mini Assignment | XJTLU 2024–25

---
## Section 1: Data Collection & Initial Exploration

In this section, we download historical OHLCV (Open, High, Low, Close, Volume) data for Bitcoin (BTC-USD) and Ethereum (ETH-USD) from Yahoo Finance using the `yfinance` library. The dataset spans from January 2022 to January 2025, covering approximately three years of daily price data. This period is particularly interesting as it includes the 2022 bear market, the 2023 recovery, and the 2024 bull run, providing diverse market conditions for model training and evaluation.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Libraries loaded successfully.')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

In [ ]:
# Download historical data from Yahoo Finance
btc = yf.download('BTC-USD', start='2022-01-01', end='2025-01-01', auto_adjust=True, progress=False)
eth = yf.download('ETH-USD', start='2022-01-01', end='2025-01-01', auto_adjust=True, progress=False)

# Flatten MultiIndex columns if present
if isinstance(btc.columns, pd.MultiIndex):
    btc.columns = btc.columns.get_level_values(0)
if isinstance(eth.columns, pd.MultiIndex):
    eth.columns = eth.columns.get_level_values(0)

print('BTC Data Shape:', btc.shape)
print('ETH Data Shape:', eth.shape)
print('\nBTC Data Preview:')
display(btc.head())
print('\nMissing Values (BTC):')
display(btc.isnull().sum())

In [ ]:
# Descriptive statistics
print('BTC Descriptive Statistics:')
display(btc.describe().round(2))
print('\nETH Descriptive Statistics:')
display(eth.describe().round(2))

In [ ]:
# Save combined data to CSV
import os
os.makedirs('../data', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

btc_save = btc.copy()
eth_save = eth.copy()
btc_save['Asset'] = 'BTC-USD'
eth_save['Asset'] = 'ETH-USD'
combined = pd.concat([btc_save, eth_save])
combined.to_csv('../data/crypto_data.csv')
print('Data saved to ../data/crypto_data.csv')

In [ ]:
# Figure 1: Price History with Volume
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Bitcoin & Ethereum Price History (2022–2025)', fontsize=16, fontweight='bold', y=1.01)

# BTC Price
ax1 = axes[0, 0]
ax1.plot(btc.index, btc['Close'], color='#F7931A', linewidth=1.5, label='BTC Close')
ax1.set_title('Bitcoin (BTC-USD) — Closing Price', fontsize=12, fontweight='bold')
ax1.set_ylabel('Price (USD)', fontsize=10)
ax1.legend()
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# BTC Volume
ax2 = axes[1, 0]
ax2.bar(btc.index, btc['Volume'], color='#F7931A', alpha=0.6, width=1)
ax2.set_title('Bitcoin (BTC-USD) — Trading Volume', fontsize=12, fontweight='bold')
ax2.set_ylabel('Volume', fontsize=10)
ax2.set_xlabel('Date', fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))

# ETH Price
ax3 = axes[0, 1]
ax3.plot(eth.index, eth['Close'], color='#627EEA', linewidth=1.5, label='ETH Close')
ax3.set_title('Ethereum (ETH-USD) — Closing Price', fontsize=12, fontweight='bold')
ax3.set_ylabel('Price (USD)', fontsize=10)
ax3.legend()
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# ETH Volume
ax4 = axes[1, 1]
ax4.bar(eth.index, eth['Volume'], color='#627EEA', alpha=0.6, width=1)
ax4.set_title('Ethereum (ETH-USD) — Trading Volume', fontsize=12, fontweight='bold')
ax4.set_ylabel('Volume', fontsize=10)
ax4.set_xlabel('Date', fontsize=10)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e9:.1f}B'))

plt.tight_layout()
plt.savefig('../outputs/fig1_price_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved to ../outputs/fig1_price_history.png')

---
## Section 2: Feature Engineering

This section constructs ten technical indicators from the raw OHLCV data. All indicators are computed manually using `pandas` without relying on external technical analysis libraries, which demonstrates a clear understanding of the underlying mathematical formulas.

The indicators are chosen to capture different market dynamics:
- **Momentum indicators**: RSI-14, MACD, MA_Ratio — measure the speed and direction of price movements
- **Volatility indicators**: Bollinger Band Width, 7-day Rolling Volatility — capture market uncertainty
- **Trend indicators**: MA_7, MA_30 — smooth out short-term fluctuations
- **Volume indicators**: Volume_Change — reflects market participation and conviction

The **target variable** is binary: 1 if tomorrow's closing price is higher than today's (price goes up), and 0 otherwise. This transforms the problem into a binary classification task.

In [ ]:
def compute_features(df):
    """Compute technical indicators from OHLCV data."""
    df = df.copy()

    # Returns and volatility
    df['Price_Change_Pct'] = df['Close'].pct_change()
    df['Volatility_7'] = df['Price_Change_Pct'].rolling(7).std()
    df['Volume_Change'] = df['Volume'].pct_change()

    # Moving averages
    df['MA_7'] = df['Close'].rolling(7).mean()
    df['MA_30'] = df['Close'].rolling(30).mean()
    df['MA_Ratio'] = df['MA_7'] / df['MA_30']  # momentum signal

    # RSI (14-day): Relative Strength Index
    # Formula: RSI = 100 - (100 / (1 + RS)), where RS = avg_gain / avg_loss
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))

    # MACD: Moving Average Convergence Divergence
    # MACD = EMA(12) - EMA(26); Signal = EMA(9) of MACD
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

    # Bollinger Bands: 20-day MA ± 2 standard deviations
    df['BB_Mid'] = df['Close'].rolling(20).mean()
    bb_std = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['BB_Mid'] + 2 * bb_std
    df['BB_Lower'] = df['BB_Mid'] - 2 * bb_std
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Mid']
    df['BB_Position'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

    # Target variable: 1 if tomorrow's close > today's close
    df['Target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, 0)

    return df.dropna()

btc_feat = compute_features(btc)
eth_feat = compute_features(eth)

print(f'BTC features shape: {btc_feat.shape}')
print(f'ETH features shape: {eth_feat.shape}')
print('\nNew feature columns:')
new_cols = ['Price_Change_Pct', 'Volatility_7', 'Volume_Change', 'MA_7', 'MA_30',
            'MA_Ratio', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
            'BB_Width', 'BB_Position', 'Target']
display(btc_feat[new_cols].head())

In [ ]:
# Figure 2: Technical Indicators Visualisation (BTC)
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
fig.suptitle('Bitcoin Technical Indicators (2022–2025)', fontsize=15, fontweight='bold')

plot_df = btc_feat.iloc[-400:]  # last ~400 days for visual clarity

# Panel 1: Price + Moving Averages + Bollinger Bands
axes[0].plot(plot_df.index, plot_df['Close'], color='#F7931A', linewidth=1.5, label='Close Price')
axes[0].plot(plot_df.index, plot_df['MA_7'], color='blue', linewidth=1, linestyle='--', label='MA_7')
axes[0].plot(plot_df.index, plot_df['MA_30'], color='red', linewidth=1, linestyle='--', label='MA_30')
axes[0].fill_between(plot_df.index, plot_df['BB_Upper'], plot_df['BB_Lower'],
                     alpha=0.1, color='gray', label='Bollinger Bands')
axes[0].set_ylabel('Price (USD)', fontsize=10)
axes[0].set_title('Price with Moving Averages & Bollinger Bands', fontsize=11)
axes[0].legend(loc='upper left', fontsize=8)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Panel 2: RSI
axes[1].plot(plot_df.index, plot_df['RSI_14'], color='purple', linewidth=1.2)
axes[1].axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[1].axhline(y=30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[1].axhline(y=50, color='gray', linestyle=':', alpha=0.5)
axes[1].fill_between(plot_df.index, plot_df['RSI_14'], 70,
                     where=(plot_df['RSI_14'] >= 70), alpha=0.3, color='red')
axes[1].fill_between(plot_df.index, plot_df['RSI_14'], 30,
                     where=(plot_df['RSI_14'] <= 30), alpha=0.3, color='green')
axes[1].set_ylabel('RSI (14)', fontsize=10)
axes[1].set_title('Relative Strength Index (RSI-14)', fontsize=11)
axes[1].set_ylim(0, 100)
axes[1].legend(loc='upper left', fontsize=8)

# Panel 3: MACD
axes[2].plot(plot_df.index, plot_df['MACD'], color='blue', linewidth=1.2, label='MACD')
axes[2].plot(plot_df.index, plot_df['MACD_Signal'], color='red', linewidth=1.2, label='Signal Line')
axes[2].bar(plot_df.index, plot_df['MACD_Hist'],
            color=['green' if v >= 0 else 'red' for v in plot_df['MACD_Hist']],
            alpha=0.5, width=1, label='Histogram')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].set_ylabel('MACD', fontsize=10)
axes[2].set_title('MACD (12, 26, 9)', fontsize=11)
axes[2].legend(loc='upper left', fontsize=8)

# Panel 4: Bollinger Band Width
axes[3].plot(plot_df.index, plot_df['BB_Width'], color='teal', linewidth=1.2, label='BB Width')
axes[3].fill_between(plot_df.index, plot_df['BB_Width'], alpha=0.2, color='teal')
axes[3].set_ylabel('BB Width', fontsize=10)
axes[3].set_title('Bollinger Band Width (Volatility Indicator)', fontsize=11)
axes[3].set_xlabel('Date', fontsize=10)
axes[3].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/fig2_technical_indicators.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved to ../outputs/fig2_technical_indicators.png')

---
## Section 3: Exploratory Data Analysis

Before building predictive models, we conduct exploratory data analysis (EDA) to understand the distribution of features, check for class imbalance in the target variable, and examine correlations between features. A balanced target variable (approximately 50% up, 50% down) is important for fair model evaluation, as it ensures that a naive classifier cannot achieve high accuracy simply by predicting the majority class.

In [ ]:
feature_cols = ['Price_Change_Pct', 'Volatility_7', 'Volume_Change',
                'MA_Ratio', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
                'BB_Width', 'BB_Position']

# Descriptive statistics of features
print('BTC Feature Descriptive Statistics:')
display(btc_feat[feature_cols].describe().round(4))

# Target variable distribution
print('\nBTC Target Distribution:')
btc_target_dist = btc_feat['Target'].value_counts(normalize=True)
print(f'  Down (0): {btc_target_dist[0]:.2%}')
print(f'  Up   (1): {btc_target_dist[1]:.2%}')

print('\nETH Target Distribution:')
eth_target_dist = eth_feat['Target'].value_counts(normalize=True)
print(f'  Down (0): {eth_target_dist[0]:.2%}')
print(f'  Up   (1): {eth_target_dist[1]:.2%}')

In [ ]:
# Figure 3: Correlation Heatmap
feature_cols_with_target = feature_cols + ['Target']
corr_matrix = btc_feat[feature_cols_with_target].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix — Bitcoin (BTC-USD)', fontsize=13, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/fig3_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved to ../outputs/fig3_correlation_heatmap.png')

---
## Section 4: Machine Learning Models

We train two classification models to predict the next-day price direction:

1. **Logistic Regression** — A linear baseline model that estimates the probability of price increase using a logistic function. It is interpretable and computationally efficient, serving as a benchmark.

2. **Random Forest Classifier** — An ensemble model that builds multiple decision trees and aggregates their predictions. It can capture non-linear relationships between features and the target variable, and provides feature importance scores.

**Important methodological note:** We use a **chronological train/test split** (80% training, 20% testing) rather than random shuffling. This is critical for time-series data because random splitting would cause **data leakage** — future information would leak into the training set, artificially inflating model performance. The split point corresponds to approximately October 2024.

In [ ]:
feature_cols_ml = ['Price_Change_Pct', 'Volatility_7', 'Volume_Change',
                   'MA_Ratio', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
                   'BB_Width', 'BB_Position']

def train_and_evaluate(df, asset_name):
    """Train and evaluate Logistic Regression and Random Forest models."""
    X = df[feature_cols_ml]
    y = df['Target']

    # Time-series split: 80% train, 20% test (chronological order — no data leakage)
    split_idx = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    print(f'Training period: {df.index[0].date()} to {df.index[split_idx-1].date()}')
    print(f'Testing period:  {df.index[split_idx].date()} to {df.index[-1].date()}')
    print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

    # Standardise features (required for Logistic Regression)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # Model 1: Logistic Regression
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train_sc, y_train)
    lr_pred = lr.predict(X_test_sc)
    lr_prob = lr.predict_proba(X_test_sc)[:, 1]
    lr_acc = accuracy_score(y_test, lr_pred)
    lr_auc = roc_auc_score(y_test, lr_prob)

    # Model 2: Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_prob = rf.predict_proba(X_test)[:, 1]
    rf_acc = accuracy_score(y_test, rf_pred)
    rf_auc = roc_auc_score(y_test, rf_prob)

    print(f'\n{"="*55}')
    print(f'Results for {asset_name}')
    print(f'{"="*55}')
    print(f'\nLogistic Regression — Accuracy: {lr_acc:.4f} | AUC: {lr_auc:.4f}')
    print(classification_report(y_test, lr_pred, target_names=['Down', 'Up']))
    print(f'\nRandom Forest — Accuracy: {rf_acc:.4f} | AUC: {rf_auc:.4f}')
    print(classification_report(y_test, rf_pred, target_names=['Down', 'Up']))

    return {
        'rf': rf, 'lr': lr,
        'X_train': X_train, 'X_test': X_test,
        'y_test': y_test,
        'rf_pred': rf_pred, 'lr_pred': lr_pred,
        'rf_prob': rf_prob, 'lr_prob': lr_prob,
        'rf_acc': rf_acc, 'lr_acc': lr_acc,
        'rf_auc': rf_auc, 'lr_auc': lr_auc
    }

print('Training models for Bitcoin...')
btc_res = train_and_evaluate(btc_feat, 'Bitcoin')
print('\nTraining models for Ethereum...')
eth_res = train_and_evaluate(eth_feat, 'Ethereum')

---
## Section 5: Model Evaluation & Visualisation

We evaluate model performance using three complementary metrics:

- **Accuracy**: The proportion of correct predictions. A 50% baseline represents random guessing.
- **AUC-ROC**: Area Under the Receiver Operating Characteristic curve. Values above 0.5 indicate better-than-random discrimination ability.
- **Confusion Matrix**: Shows the breakdown of true positives, true negatives, false positives, and false negatives.

We also examine **feature importance** from the Random Forest model, which indicates which technical indicators contribute most to the predictions.

In [ ]:
# Figure 4: Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Random Forest Feature Importance', fontsize=14, fontweight='bold')

for ax, res, asset, color in zip(axes,
                                  [btc_res, eth_res],
                                  ['Bitcoin (BTC-USD)', 'Ethereum (ETH-USD)'],
                                  ['#F7931A', '#627EEA']):
    importances = res['rf'].feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols_ml).sort_values(ascending=True)
    bars = ax.barh(feat_imp.index, feat_imp.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'{asset}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Importance Score', fontsize=10)
    ax.set_xlim(0, feat_imp.values.max() * 1.2)
    for bar, val in zip(bars, feat_imp.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', ha='left', fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/fig4_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 4 saved to ../outputs/fig4_feature_importance.png')

In [ ]:
# Figure 5: Model Comparison — ROC Curves & Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Model Evaluation: ROC Curves & Confusion Matrices', fontsize=14, fontweight='bold')

assets = ['Bitcoin (BTC-USD)', 'Ethereum (ETH-USD)']
results = [btc_res, eth_res]

for col, (asset, res) in enumerate(zip(assets, results)):
    # ROC Curves
    ax_roc = axes[0, col]
    for model_name, prob, color in [('Logistic Regression', res['lr_prob'], 'blue'),
                                     ('Random Forest', res['rf_prob'], 'darkorange')]:
        fpr, tpr, _ = roc_curve(res['y_test'], prob)
        auc = roc_auc_score(res['y_test'], prob)
        ax_roc.plot(fpr, tpr, color=color, linewidth=2,
                    label=f'{model_name} (AUC = {auc:.3f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
    ax_roc.set_xlabel('False Positive Rate', fontsize=10)
    ax_roc.set_ylabel('True Positive Rate', fontsize=10)
    ax_roc.set_title(f'{asset} — ROC Curves', fontsize=11, fontweight='bold')
    ax_roc.legend(loc='lower right', fontsize=9)
    ax_roc.set_xlim([0, 1])
    ax_roc.set_ylim([0, 1.05])
    ax_roc.grid(alpha=0.3)

    # Confusion Matrix (Random Forest)
    ax_cm = axes[1, col]
    cm = confusion_matrix(res['y_test'], res['rf_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                xticklabels=['Down', 'Up'],
                yticklabels=['Down', 'Up'],
                cbar=False,
                annot_kws={'size': 14, 'weight': 'bold'})
    ax_cm.set_xlabel('Predicted Label', fontsize=10)
    ax_cm.set_ylabel('True Label', fontsize=10)
    ax_cm.set_title(f'{asset} — Random Forest Confusion Matrix\n(Accuracy: {res["rf_acc"]:.3f})',
                    fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/fig5_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 5 saved to ../outputs/fig5_model_comparison.png')

---
## Section 6: Key Findings

The results summary table below consolidates the performance metrics for all four model-asset combinations.

In [ ]:
# Results Summary Table
results_summary = pd.DataFrame({
    'Asset': ['Bitcoin', 'Bitcoin', 'Ethereum', 'Ethereum'],
    'Model': ['Logistic Regression', 'Random Forest', 'Logistic Regression', 'Random Forest'],
    'Accuracy': [btc_res['lr_acc'], btc_res['rf_acc'], eth_res['lr_acc'], eth_res['rf_acc']],
    'AUC-ROC': [btc_res['lr_auc'], btc_res['rf_auc'], eth_res['lr_auc'], eth_res['rf_auc']]
})
print('Model Performance Summary:')
display(results_summary.round(4))

### Key Findings

**Finding 1 — Model Performance Near Random Baseline:** Both models achieve accuracy and AUC-ROC scores close to 0.50, which is the theoretical random baseline for binary classification. For Bitcoin, Logistic Regression achieves the highest accuracy (50.93%) and AUC (0.509), while Random Forest performs slightly below random on BTC (46.73%). This is consistent with the **Efficient Market Hypothesis (EMH)**, which posits that asset prices already reflect all available information, making systematic prediction from historical data alone extremely difficult.

**Finding 2 — Logistic Regression Outperforms Random Forest:** Contrary to the common expectation that ensemble methods outperform linear models, Logistic Regression achieves higher accuracy and AUC than Random Forest on both BTC and ETH. This suggests that the relationship between technical indicators and next-day price direction is largely linear (or near-random), and the added complexity of Random Forest does not provide meaningful benefit — and may even introduce overfitting on the training set.

**Finding 3 — Most Important Features:** For Bitcoin, `Price_Change_Pct` (14.26%) and `Volatility_7` (11.11%) are the top two features in the Random Forest model, followed by `MACD_Hist` (11.47%) and `BB_Position` (10.99%). For Ethereum, `Volatility_7` (15.38%) is the most important feature, followed by `Price_Change_Pct` (12.15%). This indicates that short-term momentum and volatility carry more predictive signal than longer-term trend indicators such as MACD or moving average ratios.

**Finding 4 — Cryptocurrency Market Efficiency:** The near-random model performance across both assets and both models provides empirical evidence that cryptocurrency markets, despite their perceived inefficiency, are difficult to predict using technical indicators alone over the 2022–2025 period. The inclusion of a bear market (2022), recovery (2023), and bull market (2024) in the dataset makes it particularly challenging for models to learn stable patterns.

---
## Section 7: Limitations & Conclusion

### Limitations

**1. Exclusive Reliance on Technical Indicators:** This study uses only price-derived technical indicators. Cryptocurrency markets are heavily influenced by sentiment data (Twitter/Reddit discussions, Fear & Greed Index), on-chain metrics (active addresses, transaction volume, miner behaviour), and macroeconomic factors (interest rates, regulatory announcements). Incorporating these additional data sources could substantially improve predictive power.

**2. Transaction Costs and Slippage:** The models do not account for trading costs, bid-ask spreads, or market impact. Even if a model achieves slightly above 50% accuracy, real-world profitability after costs is not guaranteed.

**3. Non-stationarity of Cryptocurrency Markets:** Cryptocurrency markets exhibit structural breaks and regime changes (e.g., the 2022 Terra/LUNA collapse, FTX bankruptcy). Models trained on historical patterns may not generalise to future market regimes, a fundamental challenge in financial machine learning.

**4. Sample Period Heterogeneity:** The dataset spans both a severe bear market (2022, BTC fell ~65%) and a strong bull market (2024, BTC reached new all-time highs). The statistical properties of technical indicators differ significantly across these regimes, making it difficult for a single model to capture both environments effectively.

**5. No Hyperparameter Optimisation:** The models use default or manually set hyperparameters. A systematic grid search or Bayesian optimisation could potentially improve performance, though the risk of overfitting to the test set would need to be managed carefully.

### Conclusion

This project demonstrates the application of machine learning to cryptocurrency price direction prediction using manually engineered technical indicators. The results show that both Logistic Regression and Random Forest models perform close to the random baseline (50%), consistent with the efficient market hypothesis. The most informative features are short-term momentum (`Price_Change_Pct`) and volatility (`Volatility_7`), rather than complex indicators like MACD or Bollinger Bands. While the models do not achieve practically useful prediction accuracy, the project illustrates the complete data science pipeline — from data acquisition and feature engineering to model training, evaluation, and critical interpretation of results. Future work could explore incorporating sentiment analysis, on-chain data, or more sophisticated models such as LSTM neural networks to potentially improve predictive performance.